<a href="https://colab.research.google.com/github/maivu18082005-hub/Parameter_Identification_of_PMSM_using_PINN/blob/main/04_train_pinn.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# ===== CELL 1: KẾT NỐI DRIVE & KIỂM TRA GPU =====
from google.colab import drive  # Import thư viện drive của Colab để gắn kết nối tới Google Drive
import torch                    # Import thư viện PyTorch làm framework chính cho Deep Learning
import os                       # Import thư viện os để thực hiện các thao tác kiểm tra thư mục và đường dẫn

drive.mount('/content/drive')   # Thực hiện Mount (gắn) Google Drive của bạn vào môi trường ảo Colab

# Khai báo đường dẫn thư mục dự án gốc trên Google Drive
DRIVE_PROJECT = '/content/drive/MyDrive/Parameter_Identification_of_PMSM_using_PINN'

# Khai báo đường dẫn chứa tập dữ liệu đã qua tiền xử lý (.npy và .pkl)
PROCESSED_DATA_DIR = os.path.join(DRIVE_PROJECT, 'dataset', 'processed')

# Khai báo thư mục lưu trữ trọng số mô hình tốt nhất (Checkpoints)
CHECKPOINT_DIR = os.path.join(DRIVE_PROJECT, 'checkpoints')

# Tự động tạo thư mục checkpoints trên Google Drive nếu thư mục này chưa tồn tại
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

# Kiểm tra xem Colab có GPU không: nếu có thì dùng 'cuda', nếu không thì dùng 'cpu'
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print('Thiết bị tính toán đang dùng:', device) # In ra màn hình thiết bị đang chạy (ví dụ: cuda:0)

Mounted at /content/drive
Thiết bị tính toán đang dùng: cpu


In [2]:
# ===== CELL 2: PYTORCH DATASET CHO PINN =====
from torch.utils.data import Dataset, DataLoader # Import hai lớp quản lý Dataset và nạp dữ liệu theo Batch
import numpy as np                               # Import NumPy để nạp các file lưu dưới dạng mảng .npy

class PMSMPINNDataset(Dataset):
    """
    Lớp Custom Dataset dùng để nạp Input (X), Target (Y) và Chuỗi thời gian (Time)
    phục vụ cho tính toán vi phân trong hàm Loss Vật Lý của PINN.
    """
    def __init__(self, data_dir, split='train'):
        # Nạp mảng Input đã chuẩn hóa: ud, uq, Id, Iq, nm
        self.X = np.load(os.path.join(data_dir, f'X_{split}.npy'))

        # Nạp mảng Target đã chuẩn hóa: Rs, Ld, Lq
        self.Y = np.load(os.path.join(data_dir, f'Y_{split}.npy'))

        # Nạp mảng chuỗi thời gian Time để hỗ trợ tính toán đạo hàm theo thời gian
        self.time = np.load(os.path.join(data_dir, f'Time_{split}.npy'))

        # Chuyển đổi dữ liệu từ dạng NumPy Array sang Tensor của PyTorch (kiểu float32)
        self.X = torch.tensor(self.X, dtype=torch.float32)
        self.Y = torch.tensor(self.Y, dtype=torch.float32)
        self.time = torch.tensor(self.time, dtype=torch.float32)

    def __len__(self):
        # Trả về tổng số lượng mẫu dữ liệu có trong tập dữ liệu
        return len(self.X)

    def __getitem__(self, idx):
        # Trả về bộ 3 giá trị tương ứng với chỉ số (index) idx: Input, Target và Time
        return self.X[idx], self.Y[idx], self.time[idx]

# Nạp 3 tập dữ liệu đã phân chia từ thư mục processed trên Drive
train_dataset = PMSMPINNDataset(PROCESSED_DATA_DIR, split='train')       # Tập Train (Data1, Data2, Data3)
val_dataset = PMSMPINNDataset(PROCESSED_DATA_DIR, split='validation') # Tập Validation (Data4)
test_dataset = PMSMPINNDataset(PROCESSED_DATA_DIR, split='test')         # Tập Test (Data5)

# Khởi tạo DataLoader để chia dữ liệu thành từng Batch (kích thước 128 mẫu/batch)
train_loader = DataLoader(train_dataset, batch_size=128, shuffle=True)  # Trộn ngẫu nhiên tập Train khi nạp
val_loader = DataLoader(val_dataset, batch_size=128, shuffle=False)     # Giữ nguyên thứ tự tập Validation
test_loader = DataLoader(test_dataset, batch_size=128, shuffle=False)   # Giữ nguyên thứ tự tập Test

print(f'Đã nạp Dataset PINN thành công! Train: {len(train_dataset)} mẫu.')

Đã nạp Dataset PINN thành công! Train: 90000 mẫu.


In [3]:
# ===== CELL 3: KIẾN TRÚC MẠNG PINN & PHYSICS LOSS =====
import torch.nn as nn # Import module chứa các lớp mạng Neural của PyTorch
import joblib        # Import joblib để đọc file lưu bộ chuẩn hóa Scaler

# Nạp bộ chuẩn hóa Scaler đã lưu từ bước preprocessing để giải chuẩn hóa dữ liệu khi tính phương trình vật lý
X_scaler = joblib.load(os.path.join(PROCESSED_DATA_DIR, 'X_scaler.pkl'))
Y_scaler = joblib.load(os.path.join(PROCESSED_DATA_DIR, 'Y_scaler.pkl'))

# Chuyển các giá trị mean (trung bình) và scale (độ lệch chuẩn) của bộ Scaler thành PyTorch Tensor trên GPU
x_mean = torch.tensor(X_scaler.mean_, dtype=torch.float32).to(device)
x_scale = torch.tensor(X_scaler.scale_, dtype=torch.float32).to(device)
y_mean = torch.tensor(Y_scaler.mean_, dtype=torch.float32).to(device)
y_scale = torch.tensor(Y_scaler.scale_, dtype=torch.float32).to(device)

class PMSM_PINN(nn.Module):
    """
    Kiến trúc Mạng Feedforward Neural Network (FNN)
    Gồm 5 nút đầu vào [ud, uq, Id, Iq, nm] và 3 nút đầu ra dự đoán [Rs, Ld, Lq]
    """
    def __init__(self, input_dim=5, output_dim=3, hidden_dim=128):
        super(PMSM_PINN, self).__init__()
        # Định nghĩa chuỗi lớp liên kết đầy đủ (Fully Connected Layers)
        self.network = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),   # Lớp đầu vào: 5 biến -> 128 nút ẩn
            nn.Tanh(),                          # Hàm kích hoạt Tanh tạo tính phi tuyến
            nn.Linear(hidden_dim, hidden_dim),  # Lớp ẩn thứ nhất: 128 -> 128 nút
            nn.Tanh(),                          # Hàm kích hoạt Tanh
            nn.Linear(hidden_dim, hidden_dim),  # Lớp ẩn thứ hai: 128 -> 128 nút
            nn.Tanh(),                          # Hàm kích hoạt Tanh
            nn.Linear(hidden_dim, output_dim)   # Lớp đầu ra: 128 -> 3 tham số [Rs, Ld, Lq]
        )

    def forward(self, x):
        # Hàm lan truyền tiến (Forward Pass) truyền dữ liệu qua mạng
        return self.network(x)

def compute_physics_loss(inputs_scaled, outputs_scaled, pole_pairs=4, psi_m=0.175):
    """
    Hàm tính toán sai số vi phạm phương trình vật lý PMSM (Physics-informed Loss)
    """
    # 1. Giải chuẩn hóa (Unscale) dữ liệu Input và Output về dải giá trị thực tế (vật lý)
    inputs_real = inputs_scaled * x_scale + x_mean   # Tính: Real = Scaled * Std + Mean
    outputs_real = outputs_scaled * y_scale + y_mean # Tính: Real = Scaled * Std + Mean

    # Bóc tách từng đại lượng vật lý thực từ mảng inputs_real
    ud = inputs_real[:, 0] # Điện áp trục d (V)
    uq = inputs_real[:, 1] # Điện áp trục q (V)
    Id = inputs_real[:, 2] # Dòng điện trục d (A)
    Iq = inputs_real[:, 3] # Dòng điện trục q (A)
    nm = inputs_real[:, 4] # Tốc độ cơ học (vòng/phút - rpm)

    # Bóc tách các tham số được mạng Neural dự đoán từ outputs_real
    Rs_est = outputs_real[:, 0] # Điện trở stator ước lượng (Ohm)
    Ld_est = outputs_real[:, 1] # Độ tự cảm trục d ước lượng (H)
    Lq_est = outputs_real[:, 2] # Độ tự cảm trục q ước lượng (H)

    # 2. Quy đổi tốc độ cơ học nm (rpm) sang tốc độ góc điện omega_e (rad/s)
    omega_m = nm * (2 * np.pi / 60.0) # Đổi từ rpm sang rad/s góc cơ: omega_m = nm * 2*pi / 60
    omega_e = pole_pairs * omega_m    # Đổi sang tốc độ góc điện: omega_e = p * omega_m (p=4)

    # 3. Tính toán điện áp lý thuyết ud, uq dựa trên phương trình vi phân trạng thái xác lập PMSM
    ud_calc = Rs_est * Id - omega_e * Lq_est * Iq            # Phương trình điện áp ud = Rs*id - omega_e*Lq*iq
    uq_calc = Rs_est * Iq + omega_e * (Ld_est * Id + psi_m)  # Phương trình điện áp uq = Rs*iq + omega_e*(Ld*id + psi_m)

    # 4. Tính toán sai số bình phương giữa điện áp thực tế và điện áp tính toán từ quy luật vật lý
    loss_ud = torch.mean((ud - ud_calc) ** 2) # Sai số MSE trên trục d
    loss_uq = torch.mean((uq - uq_calc) ** 2) # Sai số MSE trên trục q

    # Trả về tổng sai số vật lý
    return loss_ud + loss_uq

# Khởi tạo mô hình PINN và chuyển lên GPU
model_pinn = PMSM_PINN().to(device)
print(model_pinn) # In cấu trúc mạng Neural ra màn hình

PMSM_PINN(
  (network): Sequential(
    (0): Linear(in_features=5, out_features=128, bias=True)
    (1): Tanh()
    (2): Linear(in_features=128, out_features=128, bias=True)
    (3): Tanh()
    (4): Linear(in_features=128, out_features=128, bias=True)
    (5): Tanh()
    (6): Linear(in_features=128, out_features=3, bias=True)
  )
)


In [11]:
# ===== CELL 4: HUẤN LUYỆN 2 GIAI ĐOẠN (FINE-TUNING CHO Rs) =====
import torch.optim as optim

# 1. Nạp lại mô hình MLP đã train có Ld, Lq rất chính xác làm móng
# Thêm map_location=device để tự động map thiết bị từ GPU sang CPU nếu cần thiết
checkpoint_mlp = torch.load(os.path.join(CHECKPOINT_DIR, 'best_mlp_model.pth'), map_location=device)
model_pinn.load_state_dict(checkpoint_mlp['model_state_dict'])

# 2. Định nghĩa hàm Loss riêng biệt (Dùng Loss trọng số nhẹ hơn)
def custom_pinn_loss(outputs, targets, inputs, lambda_p=1e-4):
    # Loss dữ liệu: Giữ trọng số cân bằng để không làm méo Ld, Lq
    loss_Rs = torch.mean((outputs[:, 0] - targets[:, 0]) ** 2)
    loss_Ld = torch.mean((outputs[:, 1] - targets[:, 1]) ** 2)
    loss_Lq = torch.mean((outputs[:, 2] - targets[:, 2]) ** 2)

    # Nhân hệ số nhẹ 2.0 cho Rs (thay vì 5.0 để tránh kéo méo Ld)
    loss_data = 2.0 * loss_Rs + 1.0 * loss_Ld + 1.0 * loss_Lq

    # Physics loss
    loss_phys = compute_physics_loss(inputs, outputs)

    return loss_data + lambda_p * loss_phys

# 3. Sử dụng Learning Rate nhỏ hơn (lr=2e-4) để tinh chỉnh mịn (Fine-tune)
optimizer = optim.Adam(model_pinn.parameters(), lr=0.0002)

# Sử dụng Scheduler tự động giảm Learning Rate khi Loss đi vào vùng hội tụ
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min', factor=0.5, patience=5
)

epochs = 100
best_val_loss = float('inf')

print("=== BẮT ĐẦU FINE-TUNING PINN (2-STAGE) ===")

for epoch in range(1, epochs + 1):
    model_pinn.train()
    train_loss = 0.0

    for inputs, targets, _ in train_loader:
        inputs, targets = inputs.to(device), targets.to(device)

        optimizer.zero_grad()
        outputs = model_pinn(inputs)

        loss = custom_pinn_loss(outputs, targets, inputs, lambda_p=1e-4)
        loss.backward()
        optimizer.step()

        train_loss += loss.item() * inputs.size(0)

    train_loss /= len(train_dataset)

    # Validation
    model_pinn.eval()
    val_loss = 0.0
    with torch.no_grad():
        for inputs, targets, _ in val_loader:
            inputs, targets = inputs.to(device), targets.to(device)
            outputs = model_pinn(inputs)
            loss = custom_pinn_loss(outputs, targets, inputs, lambda_p=1e-4)
            val_loss += loss.item() * inputs.size(0)

    val_loss /= len(val_dataset)

    # Giảm Learning Rate tự động nếu val_loss không giảm thêm
    scheduler.step(val_loss)

    if epoch % 10 == 0 or epoch == 1:
        print(f"Epoch [{epoch:3d}/{epochs}] | Train Loss: {train_loss:.6f} | Val Loss: {val_loss:.6f} | LR: {optimizer.param_groups[0]['lr']:.6f}")

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        checkpoint_path = os.path.join(CHECKPOINT_DIR, 'best_pinn_model.pth')
        torch.save({
            'epoch': epoch,
            'model_state_dict': model_pinn.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'val_loss': val_loss
        }, checkpoint_path)

print(f"\n--> Đã hoàn thành 2-Stage Fine-tuning! Weights lưu tại: {CHECKPOINT_DIR}/best_pinn_model.pth")

=== BẮT ĐẦU FINE-TUNING PINN (2-STAGE) ===
Epoch [  1/100] | Train Loss: 2.042946 | Val Loss: 2.347109 | LR: 0.000200
Epoch [ 10/100] | Train Loss: 1.933091 | Val Loss: 2.420463 | LR: 0.000200
Epoch [ 20/100] | Train Loss: 1.853053 | Val Loss: 2.349393 | LR: 0.000050
Epoch [ 30/100] | Train Loss: 1.828450 | Val Loss: 2.411626 | LR: 0.000025
Epoch [ 40/100] | Train Loss: 1.818797 | Val Loss: 2.437028 | LR: 0.000006
Epoch [ 50/100] | Train Loss: 1.815679 | Val Loss: 2.415817 | LR: 0.000002
Epoch [ 60/100] | Train Loss: 1.814810 | Val Loss: 2.419051 | LR: 0.000001
Epoch [ 70/100] | Train Loss: 1.814428 | Val Loss: 2.417317 | LR: 0.000000
Epoch [ 80/100] | Train Loss: 1.814324 | Val Loss: 2.418524 | LR: 0.000000
Epoch [ 90/100] | Train Loss: 1.814298 | Val Loss: 2.418474 | LR: 0.000000
Epoch [100/100] | Train Loss: 1.814291 | Val Loss: 2.418547 | LR: 0.000000

--> Đã hoàn thành 2-Stage Fine-tuning! Weights lưu tại: /content/drive/MyDrive/Parameter_Identification_of_PMSM_using_PINN/checkpoi

In [12]:
# ===== CELL 5: ĐÁNH GIÁ PINN TRÊN TẬP TEST =====

# Nạp bản trọng số tốt nhất đã lưu trên Google Drive ra để đánh giá
checkpoint = torch.load(os.path.join(CHECKPOINT_DIR, 'best_pinn_model.pth'))
model_pinn.load_state_dict(checkpoint['model_state_dict']) # Khôi phục trạng thái trọng số
model_pinn.eval() # Chuyển mô hình sang chế độ dự đoán/đánh giá

all_preds = []   # Mảng chứa toàn bộ kết quả dự đoán từ mạng PINN
all_targets = [] # Mảng chứa toàn bộ nhãn thực tế từ mô phỏng

with torch.no_grad(): # Tắt tính toán gradient khi kiểm tra tập Test
    for inputs, targets, _ in test_loader:
        inputs = inputs.to(device)                     # Đẩy dữ liệu lên GPU
        outputs = model_pinn(inputs)                  # Mạng PINN dự đoán
        all_preds.append(outputs.cpu().numpy())       # Lưu dự đoán về CPU dưới dạng NumPy
        all_targets.append(targets.numpy())           # Lưu nhãn thực tế dưới dạng NumPy

# Ghép các Batch nhỏ thành mảng 2D hoàn chỉnh
all_preds = np.vstack(all_preds)
all_targets = np.vstack(all_targets)

# Giải chuẩn hóa (Inverse Transform) kết quả dự đoán và nhãn thực tế về đơn vị thực (Ohm và Henry)
preds_real = Y_scaler.inverse_transform(all_preds)
targets_real = Y_scaler.inverse_transform(all_targets)

# Tính sai số phần trăm tuyệt đối trung bình (MAPE) cho từng tham số: Rs, Ld, Lq
mape = np.mean(np.abs((targets_real - preds_real) / targets_real), axis=0) * 100

print("=== KẾT QUẢ ĐÁNH GIÁ PINN TRÊN TẬP TEST ===")
print(f"Sai số Rs trung bình (PINN): {mape[0]:.2f}%") # In phần trăm sai số của Rs
print(f"Sai số Ld trung bình (PINN): {mape[1]:.2f}%") # In phần trăm sai số của Ld
print(f"Sai số Lq trung bình (PINN): {mape[2]:.2f}%") # In phần trăm sai số của Lq

=== KẾT QUẢ ĐÁNH GIÁ PINN TRÊN TẬP TEST ===
Sai số Rs trung bình (PINN): 6.80%
Sai số Ld trung bình (PINN): 0.03%
Sai số Lq trung bình (PINN): 0.04%
